# COMP9517 Lab 3 — Chinese MNIST

按实验说明组织为 **Step 1–7**（第 **7** 步含原 *Step 7/8*）；`Step 2` 含 *2.1–2.4*；`Step 1` 为安装并导入（一格代码）。数据经 **Kaggle Hub** 下载（需 API 配置）。


## Step 1：安装并导入库

下一格先 `%pip install kagglehub`（装入当前内核），再 `import` OpenCV、NumPy、pandas、scikit-learn 等（本笔记不用 matplotlib）。


In [6]:
# Step 1：kagglehub → 其余依赖
%pip install -q kagglehub

from pathlib import Path

import cv2
import kagglehub
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2：下载数据集并熟悉数据

本节共 **四格代码**，顺序与下方小标题一一对应（需已配置 Kaggle API）。第 4 格仅打印每类样例路径，不作图。


In [7]:
# Step 2：下载 + 解析路径 + 读 CSV + 熟悉数据（合并原 2.1 / 2.2）
path = kagglehub.dataset_download("gpreda/chinese-mnist")

def resolve_chinese_mnist_paths(dataset_root: Path) -> tuple[Path, Path]:
    root = Path(dataset_root)
    csv_matches = sorted(root.rglob("chinese_mnist.csv"))
    if not csv_matches:
        raise FileNotFoundError(f"No chinese_mnist.csv under {root}")
    csv_path = csv_matches[0]
    img_dir = csv_path.parent / "data" / "data"
    if not img_dir.is_dir():
        jpgs = list(root.rglob("input_*.jpg"))
        if not jpgs:
            raise FileNotFoundError("No image folder with input_*.jpg")
        img_dir = jpgs[0].parent
    return csv_path, img_dir


DATASET_ROOT = Path(path)
CSV_PATH, IMG_DIR = resolve_chinese_mnist_paths(DATASET_ROOT)
print(CSV_PATH)
print(IMG_DIR)

df_all = pd.read_csv(CSV_PATH)
label_col = "code"
df_all["path"] = df_all.apply(
    lambda r: IMG_DIR / f"input_{int(r.suite_id)}_{int(r.sample_id)}_{int(r.code)}.jpg",
    axis=1,
)

C:\Users\hrole\.cache\kagglehub\datasets\gpreda\chinese-mnist\versions\7\chinese_mnist.csv
C:\Users\hrole\.cache\kagglehub\datasets\gpreda\chinese-mnist\versions\7\data\data


## Step 3：划分并分层抽样数据集

在全集上 **分层** 留出 **1,000** 张测试集；在剩余 **14,000** 张中再分层抽取 **5,000** 与 **10,000** 张训练集。训练与测试互不重叠；两次实验共用同一测试集。**下一格代码**完成划分与抽样。


In [8]:
RANDOM_STATE = 42

train_pool_df, test_df = train_test_split(
    df_all,
    test_size=1000,
    stratify=df_all[label_col],
    random_state=RANDOM_STATE,
)
train_pool_df = train_pool_df.reset_index(drop=True)


def stratified_sample_df(pool: pd.DataFrame, n_train: int, random_state: int) -> pd.DataFrame:
    placeholder = np.zeros((len(pool), 1))
    sss = StratifiedShuffleSplit(
        n_splits=1, train_size=n_train, random_state=random_state
    )
    y_pool = pool[label_col].to_numpy()
    for train_idx, _ in sss.split(placeholder, y_pool):
        return pool.iloc[train_idx].reset_index(drop=True)


train_5k_df = stratified_sample_df(train_pool_df, 5000, random_state=RANDOM_STATE)
train_10k_df = stratified_sample_df(train_pool_df, 10000, random_state=RANDOM_STATE + 1)

## Step 4：对子集进行必要的 reshape

每张 **64×64** 灰度图展平为 **4096** 维；`X` 为 `float32`，标签为 `code`。**下一格代码**从已划分子表构建 `X_train_*`、`X_test`、`y_*`。


In [9]:
def load_Xy(sub_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    X_list, y_list = [], []
    for _, row in sub_df.iterrows():
        img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(row["path"])
        X_list.append(img.reshape(-1))
        y_list.append(int(row[label_col]))
    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list)
    return X, y


X_test, y_test = load_Xy(test_df)
X_train_5k, y_train_5k = load_Xy(train_5k_df)
X_train_10k, y_train_10k = load_Xy(train_10k_df)

LABELS = np.sort(np.unique(y_test))
print("X_test:", X_test.shape, "y_test:", y_test.shape)
print("X_train_5k:", X_train_5k.shape)
print("X_train_10k:", X_train_10k.shape)


X_test: (1000, 4096) y_test: (1000,)
X_train_5k: (5000, 4096)
X_train_10k: (10000, 4096)


## Step 5：初始化分类器

**KNN**：`n_neighbors=3`；**DT**：默认；**SGD**：`max_iter=250`，其余默认。**下一格代码**定义 `make_classifiers()`（返回字典键为 `KNN` / `DT` / `SGD`）。


In [ ]:
def make_classifiers():
    return {
        "KNN": KNeighborsClassifier(n_neighbors=3),
        "DT": DecisionTreeClassifier(),
        "SGD": SGDClassifier(max_iter=250),
    }


_probe = make_classifiers()
for name, est in _probe.items():
    print(name, "->", est)


## Step 6：训练（`fit`）

仅用训练集：`fit` 得到 `models_5k` 与 `models_10k`（10k 须重新 `make_classifiers()`）。**下一格 = Step 6 代码。**


In [ ]:
# Step 6：仅在训练集上 fit
models_5k = make_classifiers()
models_10k = make_classifiers()

for name, clf in models_5k.items():
    clf.fit(X_train_5k, y_train_5k)
    print(f"[实验A 5k] {name} fit 完成")

for name, clf in models_10k.items():
    clf.fit(X_train_10k, y_train_10k)
    print(f"[实验B 10k] {name} fit 完成")


## Step 7：测试与评估（`predict` + 指标 / 混淆矩阵）

先用 Step 6 的模型对 `X_test` 做 `predict`，再与 `y_test` 算 accuracy、precision、recall、F1（macro），并打印 `confusion_matrix` 数值数组（行/列顺序与 `LABELS` 一致）。**下一格代码**完成两步。


In [ ]:
# Step 7：测试（predict）+ 评估（指标与混淆矩阵）
print("acc, prec, rec, f1")

# 实验 A：5k 训练 → 同一测试集
print("[当前] 实验A：训练集 5000 条（测试集仍 n_test=1000）")
y = models_5k["KNN"].predict(X_test)
acc = accuracy_score(y_test, y)
prec = precision_score(y_test, y, average="macro", zero_division=0)
rec = recall_score(y_test, y, average="macro", zero_division=0)
f1 = f1_score(y_test, y, average="macro", zero_division=0)
cm = confusion_matrix(y_test, y, labels=LABELS)
print("KNN", acc, prec, rec, f1)
print(cm)

y = models_5k["DT"].predict(X_test)
acc = accuracy_score(y_test, y)
prec = precision_score(y_test, y, average="macro", zero_division=0)
rec = recall_score(y_test, y, average="macro", zero_division=0)
f1 = f1_score(y_test, y, average="macro", zero_division=0)
cm = confusion_matrix(y_test, y, labels=LABELS)
print("DT", acc, prec, rec, f1)
print(cm)

y = models_5k["SGD"].predict(X_test)
acc = accuracy_score(y_test, y)
prec = precision_score(y_test, y, average="macro", zero_division=0)
rec = recall_score(y_test, y, average="macro", zero_division=0)
f1 = f1_score(y_test, y, average="macro", zero_division=0)
cm = confusion_matrix(y_test, y, labels=LABELS)
print("SGD", acc, prec, rec, f1)
print(cm)

# 实验 B：10k 训练 → 同一测试集
print("[当前] 实验B：训练集 10000 条（测试集仍 n_test=1000）")
y = models_10k["KNN"].predict(X_test)
acc = accuracy_score(y_test, y)
prec = precision_score(y_test, y, average="macro", zero_division=0)
rec = recall_score(y_test, y, average="macro", zero_division=0)
f1 = f1_score(y_test, y, average="macro", zero_division=0)
cm = confusion_matrix(y_test, y, labels=LABELS)
print("KNN", acc, prec, rec, f1)
print(cm)

y = models_10k["DT"].predict(X_test)
acc = accuracy_score(y_test, y)
prec = precision_score(y_test, y, average="macro", zero_division=0)
rec = recall_score(y_test, y, average="macro", zero_division=0)
f1 = f1_score(y_test, y, average="macro", zero_division=0)
cm = confusion_matrix(y_test, y, labels=LABELS)
print("DT", acc, prec, rec, f1)
print(cm)

y = models_10k["SGD"].predict(X_test)
acc = accuracy_score(y_test, y)
prec = precision_score(y_test, y, average="macro", zero_division=0)
rec = recall_score(y_test, y, average="macro", zero_division=0)
f1 = f1_score(y_test, y, average="macro", zero_division=0)
cm = confusion_matrix(y_test, y, labels=LABELS)
print("SGD", acc, prec, rec, f1)
print(cm)